# Evaluation setup — intentionally incomplete

This notebook loads a saved policy and the shared interfaces. It does not yet run evaluation
or compute scientific metrics. The evaluation developer owns the metrics import, inference
loop, aggregation, and statistical analysis. Use the same observation, parsing, and game
rules as training, while choosing the evaluation manifest and decoding protocol explicitly.

In [ ]:
import os
from pathlib import Path

PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / "pyproject.toml").exists():
    PROJECT_DIR = PROJECT_DIR.parent
os.chdir(PROJECT_DIR)

CHECKPOINT = PROJECT_DIR / "runs" / "smoke-seed-42" / "final"
EVALUATION_DIR = PROJECT_DIR / "runs" / "evaluation-smoke"
EVALUATION_MANIFEST = PROJECT_DIR / "tests" / "fixtures" / "animals.json"
DEVICE = "cpu"
EVALUATION_SEED = 999
EVALUATION_BATCH_SIZE = 4
EVALUATION_TEMPERATURE = 1.0
# The fixture roster is for setup only. Supply the frozen held-out manifest later.

In [ ]:
from integration.artifacts import load_json, write_json
from rl_pipeline.rollout import TRANSCRIPT_VERSION

assert (CHECKPOINT / "adapter_config.json").is_file(), "Train first or select a saved adapter."
metadata = load_json(CHECKPOINT.parent / "run.json")
assert metadata["contract_version"] == TRANSCRIPT_VERSION, "Review a changed observation protocol."
assert EVALUATION_MANIFEST.is_file(), "Select an evaluation manifest."
EVALUATION_DIR.mkdir(parents=True, exist_ok=True)
write_json(
    EVALUATION_DIR / "evaluation_setup.json",
    {
        "checkpoint": str(CHECKPOINT),
        "source_run_identity": metadata["identity"],
        "contract_version": TRANSCRIPT_VERSION,
        "manifest": str(EVALUATION_MANIFEST),
        "seed": EVALUATION_SEED,
        "batch_size": EVALUATION_BATCH_SIZE,
        "temperature": EVALUATION_TEMPERATURE,
    },
)

In [ ]:
import torch

from parser.api import ApiParserConfig, ApiQuestionParser
from parser.deterministic import QuestionIDParser
from rl_pipeline.models import TransformersPolicy, load_inference_policy

model, tokenizer = load_inference_policy(CHECKPOINT, device=DEVICE)
model.eval()
torch.manual_seed(EVALUATION_SEED)
policy = TransformersPolicy(model, tokenizer, temperature=EVALUATION_TEMPERATURE)

parser_model = metadata["run"]["parser_model"]
if parser_model:
    parser = ApiQuestionParser(
        ApiParserConfig(
            model=parser_model,
            cache_dir=str(EVALUATION_DIR / "parser_cache"),
        )
    )
else:
    parser = QuestionIDParser()
# Use torch.inference_mode() around future inference calls. No optimizer is constructed.

## Metrics import — developer TODO

Choose the metric module and frozen definitions. Parser service failures, parser output
failures, invalid policy actions, cache reuse, and incomplete evaluation coverage are
available separately in the records. Do not silently treat missing infrastructure results
as ordinary losses or omit them from coverage reporting.

In [ ]:
# TODO: Import the agreed evaluation metrics.


## Inference loop — developer TODO

Reuse `rl_pipeline.rollout.play_games` for the same public prompts, batched policy calls,
parser, environment rules, token limits, and diagnostics. Evaluation does not require a
GRPO group size or reward calculator. Assign unique game/rollout IDs and save outputs as
they complete. Select the held-out manifest, ordering, and decoding procedure explicitly.

In [ ]:
# TODO: Load the agreed evaluation games and implement the inference loop.


## Aggregation — developer TODO

Use authoritative complete episode records. Define request-level versus action-level
parser denominators and retain incomplete coverage. Join on explicit game/checkpoint IDs.

In [ ]:
# TODO: Aggregate saved predictions and diagnostic records.


## Statistical analysis — developer TODO

Define paired comparisons, independent training seeds, uncertainty, and visualizations.
This notebook deliberately contains no default tests or effect-size claims.

In [ ]:
# TODO: Implement the agreed statistical analysis and plots.
